In [14]:
import random as rd
import pygraphviz as pgv
import json
import datetime

print(datetime.datetime.now())
map_data = pgv.AGraph("map_data.dot")
entanglements = pgv.AGraph("entanglements.dot")
with open('units.json') as ujson:
    units = json.load(ujson)
with open('orders.json') as ojson:
    orders = json.load(ojson)

def compat(utype,tgtype):
    return tgtype == 'coast' or (tgtype == 'land' and utype == 'Army') or (tgtype == 'sea' and utype == 'Fleet')

def adj(origin,destination):
    return map_data.has_neighbor(origin,destination)

def supincl(unit,origin):
    return origin in unit['superposition']

#Handling order contradictions
filter_dict = {}
invalid_keys = []
for order in orders:
    if order['unit'] not in filter_dict:
        filter_dict[order['unit']] = order['type']
    elif filter_dict[order['unit']] != order['type']:
        invalid_keys.append(order['unit'])
invalid_keys = list(set(invalid_keys))
if invalid_keys:
    print("Incompatible orders issued to the following units : " + str(invalid_keys))
orders = [val for val in orders if val['unit'] not in invalid_keys]
for i in invalid_keys:
    orders.append({"type" : "H", "unit" : i, "origin" : "", "destination" : "", "score" : 0, "convoyed" : False, "pointer" : None})

#Handling invalid province specifications. MAKE SURE CONVOYS ARE SET AT THE BEGINNING OF THE JSON FILE. Holding orders must be duplicated for every province support held.
for order in orders:
    match order['type']:
        case 'S':
            if order['pointer'] == None:
                print("No matching order to support.")
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif not supincl(units[order['unit']],order['origin']):
                print(str(order['unit']) + " has no mass in " + str(order['origin']))
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif not adj(order['origin'],order['destination']) or not compat(units[order['unit']]['type'],map_data.get_node(order['destination']).attr['type']):
                print(str(order['unit']) + " cannot support a unit to " + str(order['destination']))
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif orders[order['pointer']]['type'] == 'H' and not supincl(units[orders[order['pointer']]['unit']],order['destination']):
                print("Cannot support " + str(orders[order['pointer']]['unit']) + " with no mass in " + str(order['destination']))
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif order['destination'] != orders[order['pointer']]['destination']:
                print("Invalid support (order mismatch)")
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif map_data.get_node(order['origin']).attr['type'] == 'coast' and map_data.get_node(order['destination']).attr['type'] == 'coast' and units[order['unit']]['type'] == "Fleet":
                match = 0
                for n in map_data.neighbors(order['origin']):
                    if map_data.get_node(n).attr['type'] == 'sea' and map_data.has_neighbor(n,order['destination']):
                        match += 1
                if match == 0:
                    print(str(order['unit']) + " cannot tunnel to " + str(order['destination']) + " (type mismatch over border)")
                    order['type'] = ''
                    order['origin'] = ''
                    order['destination'] = ''
        case 'T':
            if not supincl(units[order['unit']],order['origin']):
                print(str(order['unit']) + " has no mass in " + str(order['origin']))
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif not compat(units[order['unit']]['type'],map_data.get_node(order['destination']).attr['type']):
                print(str(order['unit']) + " cannot tunnel to " + str(order['destination']) + " (type mismatch over tile)")
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif not adj(order['origin'],order['destination']) and not order['convoyed']:
                print(str(order['unit']) + " cannot tunnel to " + str(order['destination']))
                order['type'] = ''
                order['origin'] = ''
                order['destination'] = ''
            elif map_data.get_node(order['origin']).attr['type'] == 'coast' and map_data.get_node(order['destination']).attr['type'] == 'coast' and units[order['unit']]['type'] == "Fleet":
                match = 0
                for n in map_data.neighbors(order['origin']):
                    if map_data.get_node(n).attr['type'] == 'sea' and map_data.has_neighbor(n,order['destination']):
                        match += 1
                if match == 0:
                    print(str(order['unit']) + " cannot tunnel to " + str(order['destination']) + " (type mismatch over border)")
                    order['type'] = ''
                    order['origin'] = ''
                    order['destination'] = ''
        case 'C':
            if order['pointer'] == None:
                print("No matching order to convoy.")
                order['type'] = ''
            elif not orders[order['pointer']]['convoyed']:
                print("Invalid convoy (order mismatch)")
                order['type'] = ''
            elif not supincl(units[order['unit']],orders[order['pointer']]['origin']):
                print("Invalid convoy : " + str(order['unit']) + " and " + str(orders[order['pointer']]['unit']) + " do not overlap")
                order['type'] = ''
                orders[order['pointer']]['convoyed'] = False
        case 'M':
            if units[order['unit']]['mflag']:
                print("Cannot measure " + str(order['unit']) + " two turns in a row")
                order['type'] = ''
                order['destination'] = ''
                units[order['unit']]['mflag'] = False
            elif not supincl(units[order['unit']],order['destination']):
                print(str(order['unit']) + " has no mass in " + str(order['destination']))
                order['type'] = ''
                order['destination'] = ''

#Handling cut supports and attributing support values to their corresponding order + support-induced entanglements
atkd_provinces = []
for order in orders:
    if order['type'] == 'T':
        atkd_provinces.append(order['destination'])
for order in orders:
    if order['type'] == 'S' and (order['origin'] not in atkd_provinces):
        sup_value = units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
        orders[order['pointer']]['score'] += sup_value
        entanglements.add_edge(orders[order['pointer']]['unit'],order['unit'])
    elif order['type'] == 'S' and (order['origin'] in atkd_provinces):
        print(str(order['unit']) + "'s support from " + str(order['origin']) + " was cut")

#Handling holds
for order in orders:
    if order['type'] == 'H':
        for attack in orders:
            if (attack['type'] == 'T' or attack['type'] == 'M') and supincl(units[order['unit']],attack['destination']):
                attack['score'] -= order['score'] + units[order['unit']]['superposition'].count(attack['destination']) * (100 // len(units[order['unit']]['superposition']))

#print(entanglements)
#Handling tunnel and measuring success
success_keys = []
for order in orders:
    if order['type'] == 'T' or order['type'] == 'M':
        order['score'] += units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
        roll = rd.randint(1,100)
        result = order['score'] - roll
        if result < 0:
            print(str(order['unit']) + " " + str(order['type']) + " from " + str(order['origin']) + " to " + str(order['destination']) + " attempt failed with success rate = " + str(order['score']) + " from roll = " + str(roll))
            if order['type'] == 'M' : units[order['unit']]['superposition'] = list(filter((order['destination']).__ne__, units[order['unit']]['superposition']))
        else:
            print(str(order['unit']) + " " + str(order['type']) + " from " + str(order['origin']) + " to " + str(order['destination']) +  " attempt succeeded with success rate = " + str(order['score']) + " from roll = " + str(roll))
            success_keys.append(order)
            if order['type'] == 'M' and order['unit'] in entanglements.nodes():
                print(str(entanglements.neighbours(order['unit'])) + " need to be measured as a result of entanglement with " + str(order['unit']))

#Handling bounces
provinces = map_data.nodes()
provinces = {value: 0 for value in provinces}
for unit in units.keys():
    for n in units[unit]['superposition']:
        provinces[n] += 100 // len(units[unit]['superposition'])
invasion = []
bounces = []
for i, order_i in enumerate(success_keys):
    for order_j in success_keys[i+1:]:
        weight_i = 100 // (1 + len(units[order_i['unit']]['superposition']))
        weight_j = 100 // (1 + len(units[order_j['unit']]['superposition']))
        if (order_i['destination'] == order_j['destination']) and ((weight_i + weight_j + provinces[order_i['destination']]) > 100):
            print(str(order_i['unit']) + " and " + str(order_j['unit']) + " bounced in " + str(order_i['destination']))
            bounces.append(order_i['unit'])
        else:
            invasion.append((order_i['destination'],order_i['unit'],order_i['type']))                        

invasion = set(invasion)
bounces = set(bounces)
#Handling retreats
for province, immune, otype in invasion:
    if immune not in bounces:
        if otype == 'M': 
            units[immune]['superposition'] = [province]
            units[immune]['mflag'] = True
        elif otype == 'T' :
            units[immune]['superposition'].append(province)

provinces = map_data.nodes()
provinces = {value: 0 for value in provinces}
for unit in units.keys():
    for n in units[unit]['superposition']:
        provinces[n] += 100 // len(units[unit]['superposition'])

dislodges = []
for province, immune, otype in invasion:
    for unit in units.keys():
        if (unit not in dislodges) and (unit != immune) and (province in units[unit]['superposition']) and (provinces[province] > 100):
            print(str(unit) + " was dislodged from " + str(province))
            units[unit]['superposition'] = list(filter((province).__ne__, units[unit]['superposition']))
            dislodges.append(unit)

for unit in units.keys():
    if not units[unit]['superposition']:
        print(str(unit) + " must be disbanded")
        units[unit]['type'] = "TO BE REMOVED"
reserve = 100*len(units.keys())
for key in provinces:
    reserve -= provinces[key]
print("Reserve at " + str(reserve) + "% capacity")
print("Province percentages : ", provinces)

2026-07-24 18:14:57.332187
5B M from Lit to Lit attempt succeeded with success rate = 50 from roll = 14
4B M from Cze to Cze attempt failed with success rate = 50 from roll = 72
Reserve at 4% capacity
Province percentages :  {'Ice': 66, 'NAO': 33, 'NWG': 0, 'IRI': 0, 'Ire': 66, 'CEL': 50, 'MAO': 50, 'BAR': 0, 'NTH': 0, 'Nor': 100, 'Stp': 0, 'Hol': 0, 'Sct': 33, 'Yor': 0, 'Lon': 50, 'ENG': 0, 'Bel': 0, 'Swe': 0, 'SKA': 0, 'Fin': 0, 'BOT': 0, 'Est': 0, 'Lat': 0, 'Mos': 0, 'Den': 0, 'BAL': 0, 'HEL': 0, 'Min': 50, 'Lit': 50, 'Vol': 0, 'Kiv': 100, 'Bre': 0, 'War': 100, 'Pom': 0, 'Ter': 0, 'Sil': 33, 'Cze': 0, 'Ber': 100, 'Bav': 33, 'Rhi': 33, 'Swi': 66, 'Bur': 33, 'Wal': 50, 'Bri': 50, 'Pic': 50, 'BIS': 0, 'Por': 0, 'WES': 0, 'Mor': 0, 'And': 0, 'Gas': 50, 'Cas': 0, 'Orl': 50, 'Cat': 0, 'Alg': 0, 'Tun': 100, 'Lib': 0, 'TYS': 0, 'ION': 0, 'Sic': 0, 'EAS': 0, 'Egy': 0, 'Nap': 100, 'SAR': 0, 'LYO': 50, 'Rom': 0, 'Tus': 0, 'Pie': 0, 'Alb': 50, 'ADR': 50, 'AEG': 0, 'CRE': 0, 'RED': 0, 'Leb': 0, 

In [15]:
with open("units.json", "w") as f:
    json.dump(units, f)
entanglements.write("entanglements.dot")